<a href="https://colab.research.google.com/github/AyseNil-star/AyseNil-star/blob/main/week04_predicting_structure_ipynb_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Predicting the Antibody, and Trusting the Prediction

**Computational Antibody Engineering**

Weeks 1–3 worked from crystal structures. You had the answer key. **Today the answer key goes away.**

You will predict Fv structures from sequence alone, then — because we happen to have crystal
structures for these particular antibodies — measure exactly how wrong each prediction is. That is a
luxury you will not have on a real project, which is the whole reason today matters:

> **When you have no crystal structure, what tells you whether to trust the model?**

The model ships a confidence estimate. Today you find out what that estimate is worth.

---
### Today's tool
**ABodyBuilder2**, from the ImmuneBuilder package. It predicts an Fv structure from heavy and light
chain sequences in a few seconds on a CPU, and reports a **per-residue predicted error** alongside the
coordinates. Antibody-specific predictors like this exist because the Fv is a highly conserved scaffold
carrying six variable loops — a much narrower problem than general protein folding, and one you can
solve fast and well by specialising on it.

## 0. Setup

Bigger install than usual: ANARCI (numbering), ImmuneBuilder (the predictor), and OpenMM + PDBFixer,
which ImmuneBuilder uses to relax the predicted structure into clean geometry.

**This cell takes 2–4 minutes**, and the first prediction downloads ~4 model weight files on top of
that. Start it now and read section 1 while it runs.

In [ ]:
!apt-get install -qq hmmer > /dev/null 2>&1
%pip install -q anarci ImmuneBuilder openmm pdbfixer biopython pandas matplotlib

import warnings
warnings.filterwarnings("ignore")

import shutil
assert shutil.which("hmmscan"), "hmmscan not found - re-run this cell"

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from Bio.PDB import PDBParser, Superimposer
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from anarci import number
from ImmuneBuilder import ABodyBuilder2
import urllib.request, os, time

parser = PDBParser(QUIET=True)
STANDARD = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO "
               "SER THR TRP TYR VAL".split())
IMGT_CDR = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}

print("Ready.")

## 1. Predict one antibody

Start with trastuzumab, whose crystal structure you have known since week 1.

The first call downloads model weights — expect a wait. Afterwards each prediction takes a few seconds.

In [ ]:
predictor = ABodyBuilder2()     # downloads weights on first use

VH = ("EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKG"
      "RFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS")
VL = ("DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSR"
      "SGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK")

t0 = time.time()
prediction = predictor.predict({"H": VH, "L": VL})
print(f"prediction took {time.time() - t0:.1f} s")

prediction.save("trastuzumab_pred.pdb")
print("saved trastuzumab_pred.pdb")

Two things about that output are worth pausing on.

**It was fast.** Seconds, on a CPU, for a structure that would take a crystallographer months. That
speed is what makes antibody engineering a computational discipline at all — you can afford to predict
thousands of candidates.

**It carries a confidence estimate.** ImmuneBuilder writes a **predicted error in ångströms** into the
B-factor column of the PDB file, per residue. Let's look at where the model says it is unsure.

In [ ]:
pred = parser.get_structure("p", "trastuzumab_pred.pdb")[0]


def region_of(imgt_position):
    for name, (lo, hi) in IMGT_CDR.items():
        if lo <= imgt_position <= hi:
            return name
    return "framework"


rows = []
for chain in pred:
    for r in chain:
        if r.id[0] == " " and "CA" in r:
            rows.append({"chain": chain.id, "imgt": r.id[1],
                         "region": region_of(r.id[1]),
                         "predicted_error": round(r["CA"].get_bfactor(), 2)})

conf = pd.DataFrame(rows)
print("Mean predicted error by region (A):")
print(conf.groupby("region").predicted_error.mean().round(2).to_string())
print("\nThe five residues the model is least sure about:")
print(conf.nlargest(5, "predicted_error").to_string(index=False))

**The model says CDR-H3 is its weak spot** — and nothing else comes close. That is the right
intuition (week 2: H3 is the only CDR with no canonical conformations, because of junctional
diversity). The model has learned something true about antibody biology.

But "the model says" is not evidence. Let's check it against reality.

## 2. Comparing a prediction to a crystal structure

To measure error you need to know **which predicted residue corresponds to which crystal residue.**
That is not trivial: the prediction is IMGT-numbered from position 1, while `1N8Z` uses its own
sequential numbering starting at its own residue 1, and the two do not line up.

This is exactly the problem week 2 existed to solve. We number the crystal chain with ANARCI under
IMGT, and then match prediction to crystal **by IMGT position**, not by raw residue number.

Without a shared coordinate system there is no RMSD. Numbering is not bookkeeping; it is the thing
that makes the comparison possible at all.

In [ ]:
def download(pdb_id):
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return parser.get_structure(pdb_id, path)[0]


def imgt_residues(model, chain_id):
    """Map IMGT position -> residue object, for one antibody chain of a crystal structure."""
    residues = [r for r in model[chain_id] if r.id[0] == " "]
    seq = "".join(index_to_one(three_to_index(r.get_resname()))
                  if r.get_resname() in STANDARD else "X" for r in residues)
    numbering, _ = number(seq, scheme="imgt")
    if numbering is None:
        return None
    occupied = [(p, a) for (p, ic), a in numbering if a != "-"]
    return {p: residues[i] for i, (p, a) in enumerate(occupied)}


crystal = download("1N8Z")
cryst_H = imgt_residues(crystal, "B")
cryst_L = imgt_residues(crystal, "A")
pred_H = {r.id[1]: r for r in pred["H"] if r.id[0] == " "}
pred_L = {r.id[1]: r for r in pred["L"] if r.id[0] == " "}

print(f"crystal heavy: {len(cryst_H)} IMGT positions, predicted heavy: {len(pred_H)}")
print(f"crystal light: {len(cryst_L)} IMGT positions, predicted light: {len(pred_L)}")

# Never trust a correspondence you have not checked.
mismatches = [p for p in set(cryst_H) & set(pred_H)
              if cryst_H[p].get_resname() != pred_H[p].get_resname()]
print(f"\nresidue-identity mismatches in the mapping: {len(mismatches)}")
print("If that is not zero, the alignment is wrong and every RMSD below is meaningless.")

### How to measure error

We superpose the prediction onto the crystal **using framework atoms only**, then measure deviation
everywhere.

Why framework only? Because superposing on everything would spread the CDR error into the framework
and flatter the model. Aligning on the rigid, conserved scaffold and *then* asking how far the loops
have moved is the question we actually care about: given that the framework is right, are the loops
in the right place?

In [ ]:
def compare(crystal_model, h_chain, l_chain, predicted_file):
    """Superpose a prediction on the crystal framework; return per-residue deviations."""
    p = parser.get_structure("p", predicted_file)[0]
    cryst = {"H": imgt_residues(crystal_model, h_chain),
             "L": imgt_residues(crystal_model, l_chain)}
    predicted = {c: {r.id[1]: r for r in p[c] if r.id[0] == " "} for c in ("H", "L")}

    pairs = []
    for ch in ("H", "L"):
        for pos in sorted(set(cryst[ch]) & set(predicted[ch])):
            a, b = cryst[ch][pos], predicted[ch][pos]
            if "CA" in a and "CA" in b and a.get_resname() == b.get_resname():
                pairs.append((ch, pos, region_of(pos), a["CA"], b["CA"]))

    framework = [(a, b) for _, _, reg, a, b in pairs if reg == "framework"]
    sup = Superimposer()
    sup.set_atoms([a for a, b in framework], [b for a, b in framework])
    sup.apply([b for _, _, _, _, b in pairs])          # move the prediction onto the crystal

    return pd.DataFrame([{
        "chain": ch, "imgt": pos,
        "region": "framework" if reg == "framework" else f"{ch}-{reg}",
        "predicted_error": b.get_bfactor(),
        "actual_error": a - b,
    } for ch, pos, reg, a, b in pairs])


tras = compare(crystal, "B", "A", "trastuzumab_pred.pdb")

print("trastuzumab: RMSD by region (A)")
rmsd = tras.groupby("region").actual_error.apply(lambda s: np.sqrt((s ** 2).mean()))
print(rmsd.round(2).to_string())

### YOUR TURN — Question 1

From the table you just produced:

1. What is the framework RMSD? Crystallographic coordinate error is typically a few tenths of an
   ångström — how does the prediction compare to that?
2. Which region is worst? By what factor does it exceed the framework?
3. The model predicted CDR-H3 would be its weak region. Was it right about *which* region?
4. Was it right about *how much*? Compare the model's mean predicted error for CDR-H3 (section 1) to
   the actual CDR-H3 RMSD you just computed.

Question 4 is the one that matters. Hold your answer — section 4 tests it properly.

*Double-click and write your answers here.*

1. The framework RMSD is about 0.5 Å.
This means the predicted antibody framework is very similar to the crystal structure. Smaller RMSD means a better prediction.
2.CDR-H3 is usually the hardest region to predict.
If its RMSD is 3 Å and the framework RMSD is 0.5 Å:

CDR-H3 has 6 times more error than the framework.
3.Yes! The model predicted that CDR-H3 would be the weakest region, and the actual results generally supported that prediction.
4.Not necessarily. The model predicted an error, but the actual error could be much larger.
We compare these two numbers from our table:
- Predicted error: What the model expected.
- Actual RMSD: How far the prediction was from the crystal structure.
If the actual RMSD is much larger, the model underestimated its error.

## 3. Check your inputs before you trust your outputs

We are about to run a panel. Before that, a piece of routine hygiene that is easy to skip and
expensive to skip.

Crystal structures have **missing residues**. Disordered loops produce no interpretable density, so
the crystallographer leaves them out of the model. If you build a sequence by reading the residues
present in the coordinates, a gap silently becomes a **deletion**: you hand the predictor a sequence
that no real antibody has, and it dutifully predicts a structure for it.

Nothing errors. You just get a confident prediction of the wrong molecule.

In [ ]:
CANDIDATES = {
    "trastuzumab":  ("1N8Z", "B", "A"),
    "pertuzumab":   ("1S78", "D", "C"),
    "b12":          ("1HZH", "H", "L"),
    "CR6261":       ("3GBN", "H", "L"),
    "murine_4G6K":  ("4G6K", "H", "L"),
    "bevacizumab":  ("1BJ1", "H", "L"),
    "Fab_1FBI":     ("1FBI", "H", "L"),
    "Fab_2FJG":     ("2FJG", "H", "L"),
}


def v_domain_gaps(model, chain_id, n_check=120):
    """Breaks in residue numbering across the variable domain."""
    residues = [r for r in model[chain_id] if r.id[0] == " "][:n_check]
    nums = [r.id[1] for r in residues]
    return [(nums[i], nums[i + 1]) for i in range(len(nums) - 1)
            if nums[i + 1] > nums[i] + 1]


usable = {}
for name, (pdb_id, h, l) in CANDIDATES.items():
    m = download(pdb_id)
    gaps = {"H": v_domain_gaps(m, h), "L": v_domain_gaps(m, l)}
    bad = {c: g for c, g in gaps.items() if g}
    if bad:
        missing = sum(hi - lo - 1 for g in bad.values() for lo, hi in g)
        print(f"{name:<14}{pdb_id}  EXCLUDED - {missing} residues missing from the V domain: {bad}")
    else:
        usable[name] = (pdb_id, h, l)
        print(f"{name:<14}{pdb_id}  ok")

print(f"\n{len(usable)} of {len(CANDIDATES)} antibodies usable.")

**CR6261 is excluded, and it is worth seeing why it had to be.**

Its light chain is missing a 17-residue stretch inside the variable domain. Had we not checked, we
would have predicted a structure for a sequence with a 17-residue hole in it, compared that to the
crystal, and got a large RMSD — which we would then have interpreted as *the predictor failing*. It
would have been us failing.

This is the most common way benchmarks go wrong: not a broken model, but a broken comparison. When a
tool performs surprisingly badly on one case, suspect your inputs before you suspect the tool.

## 4. The benchmark

Now predict all seven and measure the error on each.

In [ ]:
results, per_residue = [], []

for name, (pdb_id, h_chain, l_chain) in usable.items():
    m = download(pdb_id)
    cryst = {"H": imgt_residues(m, h_chain), "L": imgt_residues(m, l_chain)}

    seqs = {c: "".join(index_to_one(three_to_index(cryst[c][p].get_resname()))
                       for p in sorted(cryst[c])) for c in ("H", "L")}

    out = predictor.predict(seqs)
    out.save(f"pred_{name}.pdb")

    df = compare(m, h_chain, l_chain, f"pred_{name}.pdb")
    df["antibody"] = name
    per_residue.append(df)

    row = {"antibody": name, "pdb": pdb_id}
    row.update(df.groupby("region").actual_error
                 .apply(lambda s: round(float(np.sqrt((s ** 2).mean())), 2)).to_dict())
    results.append(row)
    print(f"  {name:<14} done")

bench = pd.DataFrame(results)
per_residue = pd.concat(per_residue, ignore_index=True)

order = ["antibody", "pdb", "framework", "H-CDR1", "H-CDR2", "H-CDR3",
         "L-CDR1", "L-CDR2", "L-CDR3"]
bench = bench[order]
bench

In [ ]:
print("Mean RMSD across the panel (A):\n")
means = bench[order[2:]].mean().round(2).sort_values()
for region, value in means.items():
    bar = "#" * int(value * 12)
    print(f"  {region:<12}{value:>5.2f}  {bar}")

### The headline result

**The framework is easy. CDR-H3 is not.**

Framework RMSD lands around half an ångström — genuinely excellent, close to the coordinate
uncertainty of the crystal structures themselves. The conserved scaffold is a solved problem.

CDR-H3 is several times worse, and it is worse in a way that matters: CDR-H3 usually sits at the
centre of the paratope. In week 1 it was one of the two largest contributors to the trastuzumab
interface. So the model is most wrong exactly where binding happens.

Notice also that the error is **not** uniform across antibodies. Look at the spread in the H-CDR3
column — some are under 1 Å, some over 3 Å. An average hides this. Which brings us to the real
question of the week.

## 5. Does the model know when it is wrong?

This is the question that decides whether any of this is usable.

If predicted error tracked actual error, you could screen predictions without ever needing a crystal
structure: trust the confident ones, flag the rest. If it does not, the confidence estimate is
decoration.

We have both numbers for every residue in the panel. Plot them against each other.

In [ ]:
r = np.corrcoef(per_residue.predicted_error, per_residue.actual_error)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for region, grp in per_residue.groupby("region"):
    ax.scatter(grp.predicted_error, grp.actual_error, s=9, alpha=0.5, label=region)
lim = max(per_residue.predicted_error.max(), 3)
ax.plot([0, lim], [0, lim], "k--", lw=1, label="perfect calibration")
ax.set_xlabel("predicted error (A)")
ax.set_ylabel("actual error (A)")
ax.set_title(f"Confidence vs. reality   (Pearson r = {r:.2f}, n = {len(per_residue)})")
ax.legend(fontsize=7, markerscale=2)
ax.set_xlim(0, lim)

ax = axes[1]
summary = per_residue.groupby("region")[["predicted_error", "actual_error"]].mean()
summary.plot.bar(ax=ax, rot=45)
ax.set_ylabel("mean error (A)")
ax.set_title("Predicted vs. actual error, by region")
plt.tight_layout()
plt.show()

summary["ratio_actual_over_predicted"] = (summary.actual_error / summary.predicted_error)
print(summary.round(2).to_string())

### Read this carefully — there are two separate findings

**Finding 1: the confidence estimate is informative.** The correlation is real and positive. Residues
the model flags as uncertain genuinely do tend to be worse. Ranked usage — "which part of this
prediction is least reliable?" — is supported by the data. The model does partly know what it does not
know.

**Finding 2: it is systematically overconfident.** Look at the ratio column. Actual error exceeds
predicted error in **every single region**, by a factor of roughly two to five. This is not noise; it
is a consistent bias. The dashed line on the left plot is perfect calibration, and essentially the
entire cloud sits above it.

So the correct way to use this number is **as a relative ranking, not as an absolute error bar.** A
residue with predicted error 0.6 Å is a worse bet than one at 0.2 Å — that comparison is sound. But
reading "0.6 Å" as *"this residue is within 0.6 Å"* will mislead you most of the time.

Confidence scores tell you what a model has seen before. They do not measure truth. You will meet this
again in week 6 with ipTM, and it will matter more there, because the stakes are higher.

## 6. The case that should make you uncomfortable

Averages and correlations can hide the failure mode that actually ends projects. Let's find the worst
single residue in the whole panel and ask what the model thought of it.

In [ ]:
worst = per_residue.nlargest(8, "actual_error")[
    ["antibody", "chain", "imgt", "region", "predicted_error", "actual_error"]]
print("The eight worst residues in the panel:\n")
print(worst.round(2).to_string(index=False))

w = per_residue.loc[per_residue.actual_error.idxmax()]
print(f"\nWorst single residue: {w.antibody}, {w.region}, IMGT {int(w.imgt)}")
print(f"  model predicted it would be off by  {w.predicted_error:.2f} A")
print(f"  it was actually off by              {w.actual_error:.2f} A")
print(f"  underestimate factor:               {w.actual_error / w.predicted_error:.0f}x")

### Sit with that.

The model placed a residue roughly **15 Å** from where it belongs — that is not a subtle error, it is
most of the width of a binding site — while reporting a predicted error of about **0.2 Å**, which is
among the *most confident* values it assigns anywhere.

This is not the model being uncertain and unlucky. It is the model being **maximally confident and
catastrophically wrong.**

And notice where it happened: CDR-H2, not CDR-H3. The region everyone watches is H3. This failure is
in a loop that the panel-level averages say is basically fine, in an antibody whose framework RMSD is
a perfectly respectable 0.67 Å. Every summary statistic we computed says this prediction is good.

**The practical consequences:**

- A per-residue confidence score cannot be used as a safety guarantee. It has no floor.
- Summary statistics hide catastrophic single-residue failures. Always look at the distribution's tail,
  not just its mean.
- If a design decision depends on one residue's position, a predicted structure alone is not sufficient
  evidence — no matter how confident it looks.

This is the honest state of the field, not a flaw in this particular tool. ABodyBuilder2 is a good
model, and this is what a good model looks like on a bad day.

## 7. One more problem: we cheated

There is a flaw in everything above, and it is not in the code.

Machine-learning models are trained on data. ABodyBuilder2 was trained on antibody structures from
SAbDab — the database of solved antibody structures. So a fair test requires antibodies the model
**did not see during training.**

Let's check when our test structures were deposited in the PDB.

In [ ]:
print(f"{'antibody':<14}{'pdb':<7}{'deposited'}")
print("-" * 36)
for name, (pdb_id, _, _) in usable.items():
    with open(f"{pdb_id}.pdb") as fh:
        header = next(l for l in fh if l.startswith("HEADER"))
    print(f"{name:<14}{pdb_id:<7}{header[50:59].strip()}")

**Every structure in our benchmark was deposited between 1996 and 2012** — many years before this
model was trained. They are almost certainly in its training set.

So what did we measure? Partly prediction, partly **recall**. The RMSDs above are optimistic, and we
have no way from these numbers alone to say by how much.

This is **data leakage**, and it is the most common reason published benchmarks overstate performance.
It is rarely deliberate. It happens because the obvious test cases are the famous, well-studied,
long-solved structures — which are exactly the ones every model has already memorised.

**What a fair benchmark would require:**

1. Structures released *after* the model's training cutoff.
2. Or a test set filtered by sequence similarity to the training set, because a near-duplicate of a
   training antibody is nearly as leaky as the antibody itself.

Take-home exercise 3 asks you to do the first of these properly. Until you have, the honest way to
report today's result is: **"framework RMSD ≈ 0.5 Å on antibodies the model has probably seen."**
That sentence is much less impressive than "framework RMSD ≈ 0.5 Å", and it is the true one.

## 8. Deliverable

Must contain: the benchmark table, the calibration plot, the worst-residue analysis, the deposition
dates, your answers to Question 1, and the paragraph below.

In [ ]:
print("=" * 68)
print("WEEK 4 SUMMARY - ABodyBuilder2 vs crystal structures")
print("=" * 68)
print(f"Antibodies benchmarked:   {len(usable)} (1 excluded for V-domain gaps)")
print(f"Residues compared:        {len(per_residue)}")
print()
print(f"Framework RMSD (mean):    {bench.framework.mean():.2f} A")
print(f"CDR-H3 RMSD (mean):       {bench['H-CDR3'].mean():.2f} A")
print(f"CDR-H3 range:             {bench['H-CDR3'].min():.2f} - {bench['H-CDR3'].max():.2f} A")
print()
print(f"Confidence correlation:   r = {r:.2f}")
print(f"Mean overconfidence:      {(per_residue.actual_error.mean() / per_residue.predicted_error.mean()):.1f}x")
print(f"Worst single residue:     {w.actual_error:.1f} A actual vs {w.predicted_error:.2f} A predicted")
print()
print("All test structures deposited 1996-2012: likely in the training set.")
print("=" * 68)

### Your paragraph

*Double-click and write 200–300 words.*

A colleague has a therapeutic antibody candidate with no experimental structure. They have run
ABodyBuilder2, and they want to use the predicted structure to decide **which three CDR residues to
mutate** in order to improve affinity. They ask you whether that is sound.

Answer them. Use your own numbers. Address:

- what the prediction is reliable for, and what it is not
- how they should and should not interpret the per-residue confidence values
- what your benchmark does *not* establish, given what you found in section 7
- whether you would proceed, and if so with what caveats or additional evidence

Do not simply say "it is unreliable." The framework really is accurate to ~0.5 Å, and that is useful.
The skill being assessed is calibrated advice: neither dismissing a good tool nor overselling it.

---

**Your answer:I would use ABodyBuilder2, but I would not choose three residues to mutate based only on its prediction.
Our results showed that the antibody framework RMSD was about 0.5 Å. This means the model predicts the main antibody structure quite accurately. However, CDR loops, especially CDR-H3, can have larger errors.The model gives a predicted error for each residue. A high value means the model is less confident about that residue. However, these values are not always accurate. In our benchmark, some actual errors were much larger than predicted.
Also, our benchmark only tested how accurately the model predicted the structure. It did not show whether the model could predict which mutations would improve binding affinity. A structurally accurate model does not automatically mean that we can identify the best mutations.
I would continue using the model, but I would first examine how the antibody binds to its target. Then, I would identify possible residues for mutation and test the selected mutations experimentally.
In conclusion, ABodyBuilder2 is a useful tool for understanding antibody structure, but I would not use it alone to choose mutations to improve affinity.**

## 9. Take-home exercises

Due before week 5.

**Exercise 1 — where does the error live? (required).**
Plot actual error against IMGT position for all seven antibodies overlaid, with CDR regions shaded.
Is the error spread evenly along the loops, or concentrated at their tips? Relate your answer to how
these loops are anchored to the framework.     The prediction error is usually larger at the tips of the CDR loops, especially CDR-H3.
The framework holds the loops in place, but the tips can move more freely. Therefore, the model can predict the framework accurately while making larger mistakes at the loop tips.

**Exercise 2 — is confidence useful as a filter? (required).**
Suppose you accept only residues with predicted error below some threshold. Sweep the threshold and
plot, for each value, (a) the fraction of residues retained and (b) the RMSD of those retained.
Then answer in one sentence: **can you use this score to select a subset you can actually trust?**
This is the practical form of the calibration question. We can use the confidence score to help identify reliable residues, but we cannot trust it completely.
For example, one residue had a predicted error of about 0.2 Å, but its actual error was about 15 Å. The model thought its prediction was accurate, but it was very wrong.
Conclusion: A low predicted error does not guarantee an accurate prediction. We must compare predicted errors with actual errors to determine whether the score is useful.

**Exercise 3 — a fair test (stretch).**
Find an antibody structure released *recently* — search [SAbDab](https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/)
or the PDB, filtering by release date, and pick one with a complete, gap-free variable domain. Run
today's pipeline on it. Is the error larger than on our 1996–2012 panel? Report honestly, and note
that a single structure is an anecdote, not a benchmark — state what you would need to make it one. We can choose a recently released antibody structure from the PDB and compare its experimental structure with the ABodyBuilder2 prediction.
First, we must check that its variable domain is complete. Then, we predict its structure and calculate the RMSD.
If the RMSD is higher than in our older antibody dataset, the prediction is less accurate for that example. However, one antibody is not enough to prove that the model performs worse on newer structures. We would need to test many recent antibodies and compare their results.

In [ ]:
# Exercise 1 - error along the sequence.
# `per_residue` has columns: antibody, chain, imgt, region, predicted_error, actual_error

# YOUR CODE HERE

In [ ]:
# Exercise 2 - confidence as a filter.

def retained_rmsd(threshold):
    """Return (fraction retained, RMSD of retained residues) at a predicted-error threshold."""
    # YOUR CODE HERE
    pass


# for t in [0.1, 0.2, 0.3, 0.5, 1.0]:
#     print(t, retained_rmsd(t))

---

### Where we go next

You can now predict one antibody and say honestly how much to trust it. That is half of what you need.

The other half is the antigen, and then the two of them together — which is much harder, because an
antibody–antigen interface is hypervariable and has no evolutionary signal to learn from.

| Week | Builds on today |
|---|---|
| **5** | Predict epitopes on the antigen — and see what happens when a predictor trained on crystal structures is fed a *predicted* one. |
| **6** | Predict the whole complex. You will meet ipTM and PAE, and every calibration lesson from today applies with higher stakes. |
| **7** | The single predicted structure is one frame of a moving molecule. |
| **13** | Data leakage, benchmark optimism, and how to fool yourself — section 7 was the preview. |

Before week 5: finish exercises 1 and 2, and commit.